# import

In [80]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Environnement agent aléatoire

In [81]:
env = gym.make("LunarLander-v3")

#récupération de l'environnement et des actions
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

# print(state_dim)
# print(action_dim)

# Test de l'agent aléatoire
state, info = env.reset(seed=42)
total_reward = 0.0
done = False
steps = 0

while not done:
    # l'agent choisit une action au hasard
    action = env.action_space.sample()
    
    # on applique l'action
    next_state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    
    # l'épisode est fini si le vaisseau s'écrase, atterrit terminated ou atteint la limite de temps truncated
    done = terminated or truncated

print(f"Score de l'agent aléatoire : {total_reward:.1f} (en {steps} étapes)")
env.close()



Score de l'agent aléatoire : -137.6 (en 115 étapes)


# Actor/Critic initialisation

In [82]:
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorCritic, self).__init__()


        # ce réseau va permettre de donner une probabiltié de la prochaine action a réaliser selon l'env du vaisseau
        self.actor = nn.Sequential(
            nn.Linear(state_dim, 64), # prend nos 8 valeurs d'entrées et les envoie dans une couche de 64 neuronnes
            nn.Tanh(), # convertit nos donneés entre -1 et 1 pour une meilleur exploitation 
            nn.Linear(64, 64), # on refournit nos données déjà entrainé sur nos 64 neuronnes dnas 64 nouveaux neurones une couche de plus
            nn.Tanh(),
            nn.Linear(64, action_dim),
            nn.Softmax(dim=-1) # garantit que la somme des probabilités vaut 1
        )
        
        # sort une seule valeur estimation de l'action
        self.critic = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        

    #  cet fonction permet de choisir le chemin que la donnée va suivre dans notre réseau de neuronne
    def forward(self, state):

        action_probs = self.actor(state)
        state_value = self.critic(state)
        return action_probs, state_value

    

# vérification de la compilation du modèle

# mise en place du modèle
dummy_model = ActorCritic(state_dim, action_dim).to(device)

# création d'un faux état (a 8 chiffres) convertit pour etre comprehensible pour un tensor 
dummy_state = torch.tensor(np.zeros(state_dim), dtype=torch.float32).to(device)

# récupération de l'action de l'actor et de l'estimation du critic
dummy_probs, dummy_value = dummy_model(dummy_state)


print(f"probabilités de l'action sortie : {dummy_probs.detach().cpu().numpy()}")
print(f"valeur de l'action : {dummy_value.item():.4f}")

probabilités de l'action sortie : [0.26655236 0.28037775 0.20839815 0.24467182]
valeur de l'action : 0.1864


# Rollout buffer

In [83]:
class RolloutBuffer:
    def __init__(self):

        #données de l'environnement
        self.states =[]
        self.actions=[]
        self.rewards= []
        self.is_terminals= []

        # nous indiquera a quel point la politique a changé
        # si par exemple après l'action il est sur a 80% de l'action faites, on comparera avec a quel point il est sur apres mise a jour de la politique genre si 90%
        # on fera 90/80 qui donnera 1.125 donc une estimation de a quel point la politique a changé
        self.logprobs= []

        # sauvegarde du v(s) du critique au moment de l'action
        self.state_values= []


    def clear(self):

        del self.states[:]
        del self.actions[:]
        del self.rewards[:]
        del self.is_terminals[:]
        del self.logprobs[:]
        del self.state_values[:]

buffer = RolloutBuffer()




# PPO initialisation -> injection de l'actor/critique

In [84]:
class PPOAgent:
    def __init__(self, state_dim,action_dim, lr= 0.0003):
        self.policy = ActorCritic(state_dim, action_dim).to(device)

        #optimiseur
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)

    def select_action(self, state, buffer):
        #conversion de notre environnement en tensor pytorch exploitable
        state = torch.FloatTensor(state).to(device)

        with torch.no_grad():
            action_probs, state_value = self.policy(state)


        # instancier les probabilités choisies par le modèle 
        dist = Categorical(action_probs)

        # va permettre de choisir l'action selon les probabilités renvoyés 
        # genre si on a 0.1 0.5 0.2 0.2 ya 50% de chance que le modele choisisse l'action 2 
        action = dist.sample()


        # remplir le buffer
        buffer.states.append(state)
        buffer.actions.append(action)
        buffer.logprobs.append(dist.log_prob(action))
        buffer.state_values.append(state_value)

        return action.item()


#vérifictation de la compilation de l'agent
ppo_agent = PPOAgent(state_dim, action_dim)



# GAE

il permet de calculer si l'action c'est révélée meilleur ou pire que ce que le critique avait prédit

In [ ]:
#  gamma indique que l'agent doit valoriser les récompenses futures presque autant que les récompenses immédiates
#   gae lambda va permettre de stabiliser lentrainement
def compute_gae(buffer, gamma=0.99, gae_lambda=0.95):

    advantages = [] # *on va ranger la note finale de chaque action + ou -
    returns = []    #  on va ranger le score réel total de l'action
    gae = 0   

    for step in reversed(range(len(buffer.rewards))):
        is_terminal = buffer.is_terminals[step] # on observe si le vaisseau a termné l'action, crashé ou pas

        if step == len(buffer.rewards) -1 or is_terminal: # si on est l'index de la fin des éléments ou que l'action est terminée
            next_value = 0 # il n'a donc plus de donneés ou c'est arreté donc plus d'état suivant, on fixe alors la valeur a 0
        else:
            # si on est pas au derniere éléments ou que l'acction n'est pas terminé alors on est en plein vol ou pleine action du vaisseau 
            # on récupère alors la valeurs que le critique nous renvoie 
            # sans .item on aurait une donnée tensor illisible donc on item pour avori un chiffre basique
            next_value = buffer.state_values[step + 1].item()

        # on récupère l'ancienne valeur du critique
        current_value = buffer.state_values[step].item()

        # si next_value est plus grand que current value alors on considère que l'action faite dans le futur est meilleur que celle du passé

        # CALCULE DE LA SURPRISE -> décalage entre l'estimation du critique et la vérité 
        # delta prend la récompense immédiate
        delta = buffer.rewards[step] + gamma * next_value - current_value

        # gae prend lisse les résultats pour avoir une estimation de la précision du critique général
        
        # vérification, si la parti est terminé on remet notre gae a 0
        if is_terminal:
            gae = 0
            
        # surprise immédiate + ce qui se passe dans le futur
        # imagine que sur un instant T on coupe le moteur, l'ia va dire "cool on économise du carburant + 5 récompenses" mais
        # sur le futur ca va litéralement tuer la trajectoire et il se crash, donc l'action a l'instant T est bonne mais par 
        # la suite on doit prendre en compte le fait vaisseau se crash.
        gae = delta + gamma * gae_lambda * gae


        
        advantages.insert(0,gae)

        # return correspond à notre prédiction du critique + le bonus ou malus de l'action 
        returns.insert(0,gae+ current_value)
    return advantages ,returns